# Documentation IA — Awalé Boissons

## 1. Objectif

Le composant IA du projet Awalé Boissons sert à analyser automatiquement les commentaires provenant de Facebook, Instagram et TikTok afin d'extraire une **Customer Voice** exploitable pour l'analyse marketing.

Les prédictions portent sur :

- `language_model` : langue du commentaire ;
- `sentiment_model` : sentiment ;
- `theme_model` : thème ;
- `product_model` : produit mentionné ;
- `is_spam_model` : détection du spam.

L'IA complète les analyses de dépenses, ventes et commandes. Elle ne remplace pas les données transactionnelles et ne permet pas, à elle seule, d'attribuer des ventes à un canal marketing.

---

## 2. Positionnement méthodologique

Le principe retenu est :

> **faits observés → signaux → limites → hypothèses → test**

Les classifications IA sont donc utilisées comme des **signaux de Customer Voice**, et non comme une preuve de causalité marketing.

Exemple :

- fait : de nombreux commentaires non-spam sont classés dans le thème `price` avec un sentiment négatif ;
- signal : le prix constitue un irritant récurrent dans la voix client ;
- limite : les commentaires ne démontrent pas qu'un canal marketing a causé cet irritant ;
- hypothèse : une campagne ou une offre peut être testée avec un mécanisme de suivi ;
- test : mesurer séparément exposition, conversion et ventes associées.

---

## 3. Données d'entrée

La source principale de l'inférence complète est :

`data/processed/social_comments_for_ai.csv`

Les commentaires sont également intégrés au pipeline dbt via :

`raw_social_comments_predictions_v2`

Puis enrichis dans :

`int_social_comments_enriched`

La jointure entre commentaires et prédictions est réalisée sur :

`comment_id`

Le grain attendu est :

> **1 ligne = 1 commentaire**

---

## 4. Modèle et stratégie d'inférence

### Modèle local

Le modèle utilisé pour le benchmark et l'inférence est :

`Qwen/Qwen2.5-0.5B-Instruct`

L'objectif est de produire des classifications structurées pour les cinq variables IA.

### Version hybride

Une version hybride a été développée afin de combiner :

- règles déterministes lorsque les conditions sont suffisamment explicites ;
- modèle local pour les cas non résolus.

Cependant, lors de l'inférence complète finale sur les 2 831 commentaires :

- commentaires traités : **2 831**
- règles seules : **0**
- modèle local utilisé : **2 831**
- `rules_complete = false` : **2 831**

Il faut donc présenter l'inférence finale comme une **inférence par modèle local**, et non comme une majorité de classifications par règles.

**Où est le prompt ?** Dans un fichier versionné : `ai/prompts/comment_classifier_v2_hybrid.txt`, lu par `ai/classify_comments_hybrid.py` (son nom et une empreinte sha256 sont affichés au début de chaque run). Le fichier `comment_classifier_v1.txt` est celui de la première version (`ai/classify_comments.py`, API OpenAI), non retenue en production. Modifier le prompt de production passe par un **nouveau fichier** (v3, …) et par un nouveau passage du benchmark humain. Le fichier de prédictions n'enregistre pas de colonne « version du prompt » : les prédictions existantes ont été produites avant cette mise en fichier.

---

## 5. Benchmark humain

Un échantillon de **50 commentaires annotés manuellement** a servi de benchmark.

Important :

> Les 50 annotations humaines constituent un jeu d'évaluation et non un jeu de fine-tuning.

Le benchmark sert à mesurer l'accord entre les prédictions et les annotations humaines.

### Résultats

| Tâche | Qwen V1 | Version hybride |
|---|---:|---:|
| Language | 72 % | 22 % |
| Sentiment | 54 % | 80 % |
| Theme | 40 % | 78 % |
| Product | 58 % | 52 % |
| Spam | 94 % | 88 % |
| Exact agreement | 8 % | 14 % |

Ces résultats montrent que la version hybride améliore certaines dimensions, notamment le sentiment et le thème, mais dégrade d'autres dimensions. Ils ne justifient donc pas une affirmation générale selon laquelle une version serait meilleure pour toutes les tâches.

### Lecture avec les taux de référence

Un taux d'accord ne se lit qu'à côté de ce que donnerait une réponse constante. Sur ces 50 commentaires, prédire toujours la classe la plus fréquente donnerait :

| Tâche | Réponse constante | Taux | Qwen V1 | Version hybride |
|---|---|---:|---:|---:|
| Language | `fr` | 72 % | 72 % | 22 % |
| Sentiment | `positive` | 54 % | 54 % | 80 % |
| Theme | `taste` | 22 % | 40 % | 78 % |
| Product | `unknown` | 42 % | 58 % | 52 % |
| Spam | `false` (jamais spam) | 94 % | 94 % | 88 % |

- **V1** atteint exactement le taux de la réponse constante sur la langue, le sentiment et le spam : sur ces trois tâches, il ne fait pas mieux qu'une réponse fixe.
- **La version hybride** améliore nettement le sentiment (+26 points), le thème (+56) et le produit (+10), mais reste **très en dessous** de la réponse constante sur la langue (22 % contre 72 %) et un peu en dessous sur le spam (88 % contre 94 %).
- **Langue :** sur le corpus complet, 2 509 commentaires sur 2 831 (88,6 %) sont prédits « en », alors que l'échantillon humain compte 1 commentaire anglais sur 50. Le champ `language_model` n'est **pas utilisable en l'état**.
- **Spam :** l'échantillon ne contient que **3 spams sur 50** ; un taux de 88 % ou 94 % ne dit presque rien sur la détection de spam. Une relecture de commentaires classés spam a montré des commentaires clients légitimes (« on ne trouve plus 😭 c'est fini ? », « vous livrez à Bingerville ? »). La détection de spam est à améliorer avant d'interpréter finement les volumes.
- **Exact agreement** (les 5 champs justes en même temps) : 14 % pour la version hybride.

Le remplacement du modèle local par un modèle plus performant est prévu ; le benchmark de 50 commentaires devra alors être rejoué et lu avec ces taux de référence.

---

## 6. Pourquoi conserver un benchmark humain ?

Le benchmark permet de détecter :

- les erreurs de classification ;
- les catégories ambiguës ;
- les limites du modèle ;
- les régressions lors d'une future modification du prompt ou du modèle.

Il doit rester séparé des données d'inférence opérationnelle.

À chaque changement important du modèle, du prompt ou des règles, le benchmark doit être rejoué.

---

## 7. Résultats de l'inférence complète

Le jeu complet contient :

- **2 831 commentaires**
- Facebook : 941
- Instagram : 1 127
- TikTok : 763

Validation du fichier de prédictions :

- 2 831 lignes ;
- `comment_id` unique ;
- 0 doublon ;
- 0 valeur NULL dans les cinq champs de prédiction.

Validation DuckDB :

- 2 831 lignes ;
- 2 831 `comment_id` uniques ;
- 0 doublon ;
- 0 valeur NULL pour langue, sentiment, thème, produit et spam ;
- modèle utilisé sur 2 831 lignes.

---

## 8. Coût et durée mesurés

Mesure réalisée le 2026-09-18 sur un échantillon de 120 commentaires tirés de
`data/processed/social_comments_for_ai.csv`, avec le code de production
(`ai/classify_comments_hybrid.py`, BATCH_SIZE=4, CPU, modèle `Qwen/Qwen2.5-0.5B-Instruct`) :

| Mesure | Valeur |
|---|---|
| Commentaires testés | 120 |
| Résolus par règles seules | 0 |
| Passés par le modèle | 120 |
| Chargement du modèle (une fois) | ~74 s |
| Débit d'inférence | 5,55 s / commentaire |
| **Extrapolation à 2 831 commentaires** | **~262 minutes (4h22)** |

Coût monétaire : nul (modèle local, aucun appel API payant). Coût réel : temps de calcul CPU.

**Ce chiffre a invalidé la cible de « 10–15 min » précédemment documentée dans le runbook et
le README** — elle n'avait jamais été mesurée. La cause : le pipeline reclassait l'intégralité de
l'historique (2 831 commentaires) à chaque run, au lieu des seuls commentaires du mois écoulé.

**Correction appliquée** : `classify_comments_hybrid.py` est maintenant incrémental. Il compare
`data/processed/social_comments_for_ai.csv` (tous les commentaires connus) aux `comment_id` déjà
présents dans `ai/evaluation/social_comments_predictions_v2_full.csv`, et ne classe que les
nouveaux. Les prédictions existantes sont conservées (jamais reclassées), les nouvelles sont
ajoutées au fichier. Validation effectuée : retrait de 15 commentaires du fichier de prédictions,
relance du script, les 15 reclassés (~84 s, 4 batches) sont revenus bit-à-bit identiques aux
prédictions d'origine (décodage glouton, `do_sample=False` → déterministe), et le fichier fusionné
retrouve exactement les 2 831 lignes.

Effet sur le cycle mensuel : un mois type apporte ~470 nouveaux commentaires (moyenne sur les 6
mois observés) → ~43 min au débit mesuré, contre 262 min pour reclasser tout l'historique. Seul le
tout premier run (aucune prédiction existante) paie le coût complet ; c'est le cas mesuré ici.
Pour forcer une reclassification complète (nouveau modèle ou nouveau prompt), supprimer
`social_comments_predictions_v2_full.csv` avant de relancer.

Piste d'optimisation restante, non implémentée : réduire `max_new_tokens` (actuellement 80,
supérieur à la taille réelle d'un JSON de réponse) pour accélérer encore le run initial.

### Reprise après interruption et sorties illisibles

Le script sauvegarde ses prédictions toutes les 5 batches (~2 minutes), de façon atomique (fichier temporaire puis remplacement). Un plantage ou un Ctrl+C ne fait perdre que le travail depuis la dernière sauvegarde : relancer reprend où le run s'est arrêté, puisque seuls les `comment_id` absents du fichier sont classés. Si la sortie du modèle est illisible, le lot est retenté commentaire par commentaire ; un commentaire qui échoue encore n'est **pas enregistré** (rien n'est deviné), le script se termine avec le code 1 et il sera retenté au lancement suivant. Ces comportements sont couverts par `tests/test_classify_checkpoint.py` (modèle simulé) et ont été vérifiés avec le vrai modèle (`Qwen/Qwen2.5-0.5B-Instruct`, versions de `requirements.txt`, CPU) : sur 24 commentaires, le processus a été arrêté brutalement après la première sauvegarde intermédiaire (20 commentaires conservés, fichier intact, aucun fichier temporaire résiduel) ; la relance n'a classé que les 4 restants, et les 24 prédictions sont **identiques** aux prédictions déjà enregistrées, sur les 7 colonnes. Ce contrôle a aussi rendu visible un point auparavant silencieux : sur ces 4 commentaires, 2 réponses du modèle étaient absentes ou hors vocabulaire (champs `product` et `theme`) et avaient reçu la valeur par défaut. Le taux sur un run complet reste à mesurer.

## 9. Résultats Customer Voice

Sur les 2 831 commentaires, 513 sont détectés comme spam. **Hors spam (2 318 commentaires)** : 1 186 positifs, 735 négatifs et 397 neutres. Spam compris, les comptes seraient de 1 236 positifs, 1 197 négatifs et 398 neutres.

Principaux thèmes (hors spam) :

| Thème | Nombre |
|---|---:|
| Taste | 680 |
| Price | 376 |
| Promotion | 313 |
| Availability | 170 |
| Packaging | 208 |
| Health | 152 |
| Delivery | 109 |
| Service | 1 |
| Other | 309 |

### Signaux principaux

En excluant le spam :

**Prix**
- 347 négatifs sur 376 commentaires du thème ;
- signal très majoritairement négatif.

**Disponibilité**
- 151 négatifs sur 170 ;
- signal très majoritairement négatif.

**Goût**
- 578 positifs ;
- 88 neutres ;
- 14 négatifs ;
- signal très majoritairement positif.

**Packaging**
- 145 positifs ;
- 61 négatifs ;
- 2 neutres.

**Promotion**
- 156 positifs ;
- 142 neutres ;
- 15 négatifs.

Ces résultats décrivent les commentaires classifiés. Ils ne constituent pas une mesure de satisfaction représentative de toute la clientèle.

---

## 10. Analyse par produit

Les produits détectés sont :

- `bissap`
- `gingembre`
- `bouye`
- `multiple`
- `unknown`
- `none`

Résultats sentiment × produit, hors spam :

| Produit | Positif | Négatif | Neutre |
|---|---:|---:|---:|
| Bissap | 211 | 41 | 10 |
| Bouye | 253 | 131 | 45 |
| Gingembre | 107 | 16 | 3 |
| Multiple | 2 | 38 | 4 |
| None | 105 | 39 | 52 |
| Unknown | 508 | 470 | 283 |

La catégorie `unknown` est importante : elle montre que l'identification automatique du produit reste une limite du système.

Il ne faut pas interpréter `unknown` comme un produit réel.

---

## 11. Intégration dans le pipeline

Le flux IA est :

```text
Social comments
      ↓
Préparation des commentaires
      ↓
Modèle IA local
      ↓
social_comments_predictions_v2_full.csv
      ↓
raw_social_comments_predictions_v2
      ↓
int_social_comments_enriched
      ↓
mart_social_monthly
      ↓
Dashboard / Customer Voice
```

Le modèle ne modifie pas les commentaires originaux.

Les prédictions sont conservées séparément puis jointes avec les données sociales via `comment_id`.

---

## 12. Contrôles qualité IA

Avant d'utiliser les prédictions, contrôler :

1. nombre de commentaires source ;
2. nombre de prédictions ;
3. unicité de `comment_id` ;
4. absence de doublons ;
5. absence de NULL dans les champs prédits ;
6. proportion de lignes effectivement traitées par le modèle ;
7. version du modèle ;
8. version du prompt/règles ;
9. durée d'exécution ;
10. coût si un modèle payant est utilisé.

Contrôle minimal :

```text
source_count == prediction_count
unique_comment_id == prediction_count
duplicate_comment_id == 0
NULL predictions == 0
```

---

## 13. Évaluation à maintenir

Le benchmark doit être suivi séparément de l'inférence complète.

Pour chaque nouvelle version :

- Accuracy par tâche ;
- Macro-F1 lorsque pertinent ;
- matrice de confusion ;
- erreurs principales ;
- exact agreement ;
- volume de données ;
- durée ;
- coût ;
- version du modèle ;
- version du prompt.

Une baisse sur une tâche ne doit pas être masquée par une amélioration sur une autre.

---

## 13 bis. Garde-fou contre les chiffres inventés, et où ne pas utiliser un modèle

**Garde-fou.** Le modèle ne produit que des catégories d'un vocabulaire fermé (langue, sentiment, thème, produit, spam) : il n'écrit jamais un nombre ni un texte libre dans le reporting. Toute réponse absente ou hors vocabulaire est ramenée à une valeur par défaut **et comptée** : le script affiche `[ATTENTION]` avec le nombre de commentaires concernés et les champs. Tous les chiffres du rapport sont des comptages SQL sur les marts (`docs/semantic_layer.yml`), jamais une sortie du modèle. `load_predictions.py` refuse un fichier avec des doublons ou des valeurs NULL, et le test dbt `int_social_comments_predictions_complete` échoue si un commentaire n'a pas de prédiction.

**Comment saurait-on qu'il a dérivé ?** Par la hausse du nombre de réponses ramenées à une valeur par défaut, par un test dbt en échec, ou par une baisse du benchmark humain rejoué (à faire à chaque changement de modèle ou de prompt, en le lisant avec les taux de référence du 5).

**Où ne pas utiliser de modèle.** (1) Pour produire ou reformuler un chiffre du reporting : chaque nombre vient d'une requête SQL. (2) Pour lire les commandes WhatsApp : des règles déterministes suffisent, sont vérifiables et sont testées (`int_whatsapp_items_multiplier`) ; un modèle y serait moins fiable et non auditable. (3) Pour attribuer une vente à un canal : ce n'est pas un problème de modèle, le suivi du clic à la commande n'existe pas.

## 14. Limites

### Représentativité

Les commentaires sociaux ne représentent pas nécessairement toute la clientèle.

### Qualité des labels

Les prédictions sont des classifications automatiques et doivent être interprétées avec les performances du benchmark.

### Produit `unknown`

Une part importante des commentaires ne permet pas une identification automatique fiable du produit.

### Spam

Les commentaires détectés comme spam doivent être exclus des analyses de Customer Voice lorsque l'objectif est d'étudier la voix client.

### Causalité

L'IA ne permet pas de conclure :

> « tel canal a généré telle vente ».

Une telle conclusion nécessite un dispositif d'attribution ou d'expérimentation approprié.

---

## 15. Bonnes pratiques pour une nouvelle version

Avant de remplacer le modèle :

1. conserver l'ancien fichier de prédictions ;
2. conserver le benchmark humain ;
3. lancer le nouveau modèle sur le même benchmark ;
4. comparer les métriques par tâche ;
5. inspecter les erreurs ;
6. vérifier les NULL et doublons ;
7. lancer l'inférence complète ;
8. vérifier le volume final ;
9. exécuter `dbt test` ;
10. documenter la version retenue.

Ne jamais comparer deux versions sur des jeux de benchmark différents sans le signaler.

---

## 16. Reproductibilité

Depuis la racine du projet :

```bash
cd <racine du projet>
```

Préparer les données sociales, puis lancer l'inférence selon le script du projet.

Après chargement dans DuckDB :

```bash
cd dbt
dbt run
dbt test
```

Le pipeline final validé au moment de la livraison compte :

- **26 modèles dbt exécutés avec succès**
- **96 tests dbt PASS**
- **0 WARN**
- **0 ERROR**

---

## 17. Utilisation dans la décision marketing

La Customer Voice peut aider à :

- identifier les irritants récurrents ;
- détecter des signaux autour du prix et de la disponibilité ;
- suivre les réactions aux promotions ;
- observer les mentions de produits ;
- alimenter les questions à tester.

Elle ne doit pas être transformée en score automatique de rentabilité d'un canal.

La recommandation budgétaire de 15 M FCFA repose donc sur plusieurs sources :

- dépenses marketing ;
- media plan ;
- mesure disponible par canal ;
- ventes ;
- qualité des données ;
- Customer Voice ;
- conditions d'instrumentation.

L'IA est un **élément de preuve complémentaire**, pas le moteur unique de la décision.

---

## 18. Résumé exécutif

Le composant IA du projet fournit une classification automatisée de 2 831 commentaires sociaux selon cinq dimensions. Un benchmark humain indépendant de 50 commentaires permet d'évaluer les performances.

La version finale a été exécutée sur l'ensemble des commentaires et validée dans DuckDB. Les résultats Customer Voice mettent notamment en évidence des signaux négatifs associés aux thèmes `price` et `availability`, ainsi qu'un signal largement positif autour du `taste`.

Ces résultats doivent rester descriptifs. Pour transformer un signal en décision marketing, le projet applique systématiquement la chaîne :

> **signal → limite → hypothèse → test mesurable**

